In [76]:
import numpy as np
import yfinance as yf
import pandas as pd
import os
import matplotlib.pyplot as plt
import torch
import cvxpy as cp
from torch.utils.data import DataLoader
from dataclasses import dataclass, asdict
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim
from pykalman import KalmanFilter

In [66]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    # super().__init__(
    #     debug=debug,
    #     **kwargs
    # )
    self.debug = debug

  def get_data(
      self,
      indices:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    macros=["^TNX", "^FVX", "DX-Y.NYB", "^TRCCRB"]
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"
    macros_path = f"macros_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = yf.download(indices, start, end, interval)
      bench_tmp = yf.download([benchmark], start, end, interval)

      bench_data = bench_tmp[["Close"]]
      macro_data = yf.download(macros, start, end, interval)[["Close"]]

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)
      macro_data.to_parquet(macros_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data = pd.read_parquet(benchmark_path)
      macro_data = pd.read_parquet(macros_path)

    benchmark = bench_data.pct_change().dropna()
    self.universe = df.columns

    return df, benchmark, macro_data

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [ ]:
class FeatureGenerator:
  def __init__(self, kalman_params, debug:bool=False, **kwargs):
    # super().__init__(
    #   debug=debug,
    #   **kwargs
    # )
    self.debug = debug
    self.kalman_params = asdict(kalman_params) if isinstance(kalman_params, KalmanParams) else kalman_params

  def garman_klass_vol(self, data):
    hl = np.square(np.log(data["High"]/data["Low"]))
    co = np.square(np.log(data["Close"]/data["Open"]))
    GK = (0.5*hl - (2*np.log(2))*co)

    return GK

  def generate_kalman_outputs(self, log_prices):
    kf_dict = {}
    dt = self.kalman_params["dt"]
    trans_mat = np.array(
      [
        [1, 1, dt],
        [0, 1, 1],
        [0, 0, 1]
      ]
    )
    obs_mat = np.array(
      [
        [1, 0, 0]
      ]
    )

    cwna_mtx = np.array(
      [
        [dt**5/20, dt**4/8, dt**3/6],
        [dt**4/8, dt**3/4, dt**2/2],
        [dt**3/6, dt**2/2, dt]
      ]
    )

    obs_cov = [[1.0]]
    init_cov = np.zeros_like(cwna_mtx)
    np.fill_diagonal(init_cov, [1]+[10]*(len(np.diag(cwna_mtx))-1))

    for col in log_prices.columns:
      init_state = np.array([log_prices[col].iloc[0], 0, 0])

      for i in ["Fast", "Mid", "Slow"]:

        trans_cov = self.kalman_params[i]*cwna_mtx
        col_name = f"{col}{i}"
        kf = KalmanFilter(
          transition_matrices=trans_mat,
          observation_matrices=obs_mat,
          transition_covariance=trans_cov,
          observation_covariance=obs_cov,
          initial_state_mean=init_state,
          initial_state_covariance=init_cov
        )

        state_means, _ = kf.filter(log_prices[col])

        denoised_values = state_means[:, 0]
        momentum_values = state_means[:, 1]
        acceleration_values = state_means[:, 2]

        kf_dict[col_name + "Velocity"] = momentum_values
        kf_dict[col_name + "Acceleration"] = acceleration_values

    return kf_dict

  def get_betas(self, X, y):
    cov_mtx = np.cov(X, y)
    cov = cov_mtx[0, 1]
    var = cov_mtx[0, 0]
    return cov / var

  def get_macro_features(self, macro_data, market_returns, window=21):
    cols = ['spread_change', 'usd_idx_change', 'cmd_idx_change']
    spread = macro_data["^TNX"] - macro_data["^FVX"]
    spread_change = spread.pct_change()
    usd_idx_change = macro_data["DX-Y.NYB"].pct_change()
    cmd_idx_change = macro_data["^TRCCRB"].pct_change()

    macro_df = pd.concat(
      [spread_change, usd_idx_change, cmd_idx_change],
      axis=1
    )
    macro_df.columns = cols

    market_tmp = market_returns.copy()

    market_tmp.columns = ['market']

    combined = pd.concat([macro_df, market_tmp], axis=1).dropna()

    del (market_returns, macro_df, market_tmp, spread, spread_change, usd_idx_change, cmd_idx_change)

    rolling_betas = pd.DataFrame(index=combined.index)

    for col in cols:
      betas = []
      for i in range(len(combined)):
        if i < window - 1:
          betas.append(np.nan)
        else:
          window_data = combined.iloc[i - window + 1 : i + 1]
          X = window_data['market'].values
          y = window_data[col].values
          betas.append(self.get_betas(X, y))

      rolling_betas[f'{col}_beta'] = betas

    return rolling_betas

  def generate_ex_R_features(self, data, window:int=21):
    ex_r_features = {}

    gk_vol = self.garman_klass_vol(data)
    for i in gk_vol.columns:
      ex_r_features["GKVol"+i] = gk_vol[i].values


    kalman_outputs = self.generate_kalman_outputs(np.log(data["Close"]))

    ex_r_features |= kalman_outputs

    ex_R_target = np.log(data["Close"].pct_change()+1).shift(-window).dropna()

    ex_r_features = pd.DataFrame(ex_r_features, index=data.index)

    return ex_r_features, ex_R_target

  def generate_cov_features(self, data, macro_data, market_returns, window=21):
    cov_features = {}

    gk_vol = self.garman_klass_vol(data)
    for i in gk_vol.columns:
      cov_features["GKVol"+i] = gk_vol[i].values

    cov_features = pd.DataFrame(cov_features, index=data.index)
    macro_betas = self.get_macro_features(
        macro_data=macro_data["Close"],
        market_returns=market_returns,
        window=window
    )

    cov_features_full = pd.concat([cov_features, macro_betas], axis=1).dropna()

    cov_target = data["Close"].pct_change()\
                              .rolling(window)\
                              .cov()\
                              .shift(-window)\
                              .dropna()

    return cov_features_full, cov_target

  def generate_features(self, data, macro_data, market_returns, window=21):
    ex_r_features, ex_r_target = self.generate_ex_R_features(data, window)
    cov_features, cov_target = self.generate_cov_features(data, macros, market_returns)

    multi_idx = cov_target.index.get_level_values(0).unique()

    cm_idx = ex_r_features.index\
                          .intersection(cov_features.index)\
                          .intersection(ex_r_target.index)\
                          .intersection(multi_idx)

    cov_target = cov_target.loc[cov_target.index.isin(cm_idx, level=0)]
    ex_r_features = ex_r_features.loc[cm_idx]
    cov_features = cov_features.loc[cm_idx]
    ex_r_target = ex_r_target.loc[cm_idx]

    return ex_r_features, ex_r_target, cov_features, cov_target




In [ ]:
@dataclass
class KalmanParams:
  dt:float = 1.0
  Fast:float = 1e-1
  Mid:float = 1e-3
  Slow:float = 1e-5
  
@dataclass
class ExRModelParams:
  d_mode:int = 64
  max_len:int = 100
  num_features:int = None # change num features to shape of the the shape of input
  nhead:int = 4
  num_layers:int = 2
  seq_len:int = 21 

In [356]:
class TemporalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0).unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :, :x.size(2), :]

In [355]:
class SpatioTemporalTransformer(nn.Module):
    def __init__(self, model_params):
        super().__init__()
        self.feature_embed = nn.Linear(model_params.num_features, model_params.d_model)

        self.pos_encoder = TemporalPositionalEncoding(
            d_model=model_params.d_model,
            max_len=model_params.seq_len
        )

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=model_params.d_model,
            nhead=model_params.nhead,
            dim_feedforward=model_params.d_model*4,
            batch_first=False
        )
        self.temporal_transformer = nn.TransformerEncoder(
            temporal_layer,
            num_layers=model_params.num_layers
        )

        spatial_layer = nn.TransformerEncoderLayer(
            d_model=model_params.d_model,
            nhead=model_params.nhead,
            dim_feedforward=model_params.d_model*4,
            batch_first=False
        )

        self.spatial_transformer = nn.TransformerEncoder(
            spatial_layer,
            num_layers=model_params.num_layers
        )

        self.output_head = nn.Linear(model_params.d_model, 1)

    def forward(self, x):
        B, N, T, F = x.shape

        x = self.feature_embed(x)
        x = self.pos_encoder(x)

        x = x.permute(2, 0, 1, 3).reshape(T, B * N, -1)
        x = self.temporal_transformer(x)

        x = x[-1, :, :].reshape(B, N, -1)
        x = x.permute(1, 0, 2)
        x = self.spatial_transformer(x)
        x = x.permute(1, 0, 2)

        predicted_log_returns = self.output_head(x).squeeze(-1)
        return predicted_log_returns

In [353]:
class GNN:
  def init(self, model_params):
    pass

  def forward(self, x):
    pass

In [258]:
class Optimizer:
  def __init__(
      self,
      turnover_penalty_weight=0.5,
      risk_aversion=3.0,
      min_weight_threshold=0.01,
      debug:bool=False, **kwargs
    ):

    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug
    self.turnover_penalty = turnover_penalty_weight
    self.risk_aversion = risk_aversion
    self.min_weight_threshold = min_weight_threshold

  def optimize_portfolio(
    self,
    ex_returns,
    fwd_cov,
    w_prev=None
  ):
    S, N = ex_returns.shape
    R = ex_returns.values

    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    ex_R = ex_returns.values @ w

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0,
    ]

    if w_prev is not None:
        w_prev_arr = np.asarray(w_prev, dtype=float)
        if w_prev_arr.shape[0] != N:
            raise ValueError(
                f"w_prev length {w_prev_arr.shape[0]} != current N={N}. "
                "Caller must align w_prev to the current filtered universe."
            )
        turnover_penalty = self.turnover_penalty_weight * cp.norm1(w - w_prev_arr)
    else:
        turnover_penalty = 0.0

    ptf_risk = cp.sqrt(cp.quad_form(w, fwd_cov))


    obj_fn = cp.Maximize(ex_R - self.risk_aversion * ptf_risk - turnover_penalty)
    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    return w

  def process_w(self, w):
    weights = np.array(w.value)
    weights[weights < self.min_weight_threshold] = 0.0

    total_w = np.sum(weights)
    if total_w > 0:
        weights_recalc = weights / total_w

    return weights_recalc

In [ ]:
class Portfolio(Optimizer, DataStore, FeatureGenerator):
  def __init__(self, recalc_window, optim_params, returns_gen_params, debug:bool=False, **kwargs):
    super().__init__(
        recalc_window,
        **optim_params,
        **returns_gen_params,
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def convert_to_simple(self, log_returns, prev_returns):
    arithm_returns = np.exp(log_returns) - 1 + 0.5*(prev_returns.std())**2
    return arithm_returns

  def generate_ex_returns(self, ex_r_data, ex_r_params):
    # collect iputs
    # load model
    # predict E[r]
    # convert r to simple with convexity adjustment
    # get prediction volatility ratio
    pass

  def generate_fwd_cov(self, cov_data, cov_params):
    # collect iputs
    # load model
    # predict Σ
    pass

  def generate_features(self):
    ex_r_data = self.generate_ex_r_features()
    cov_data = self.generate_cov_features()

    return ex_r_data, cov_data

  def optimize(self, ex_returns, fwd_cov, w_prev=None):
    w = self.optimize_portfolio(ex_returns, fwd_cov, w_prev=w_prev)
    w_clean = self.process_w(w, self.min_weight_threshold)
    return w_clean

  def train(self, ex_r_data, cov_data):
    pass

In [ ]:
class Backtest(Portfolio, DataStore):
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug